### Step 1: Mount the Google Drive

Remember to use GPU runtime before mounting your Google Drive. (Runtime --> Change runtime type).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Step 2: Open the project directory

Replace `Your_Dir` with your own path.

In [ ]:
cd Your_Dir/emg2qwerty

### Step 3: Install required packages

If you only use `decoder=ctc_greedy` (as in the examples below), you can skip KenLM-related dependencies in Colab to avoid wheel build failures (e.g., `camel-kenlm` on Python 3.12).

If you need `decoder=ctc_beam`, install KenLM separately after the base install (see optional command in the next cell).


In [ ]:
!python --version
!pip install -U pip setuptools wheel
!sed '/kenlm/d' requirements.txt > requirements_colab.txt
!pip install -r requirements_colab.txt
!pip install -e .

# Optional (only required for decoder=ctc_beam):
# !pip install camel-kenlm


### Step 4: Start your experiments

- Download/copy the dataset to `Your_Dir/emg2qwerty/data`.
- For a 40-epoch run, add `trainer.max_epochs=40`.
- Validation/test CER is printed at the end under `val_metrics` and `test_metrics` (key: `CER`).
- Hydra logs and checkpoints are saved under `logs/<date>/<time>/`.
- TensorBoard can be opened with `%tensorboard --logdir logs`.


#### Training

- The checkpoints are saved in the folder `logs`, e.g., `logs/2025-02-09/18-24-15/checkpoints/`.

In [ ]:
# Single-user training (40 epochs)
!python -m emg2qwerty.train \
  user=single_user \
  trainer.accelerator=gpu trainer.devices=1 \
  trainer.max_epochs=40


#### Testing:

- Replace `Your_Path_to_Checkpoint` with your checkpoint path.

In [ ]:
# Single-user testing
!python -m emg2qwerty.train \
  user=single_user \
  checkpoint="Your_Path_to_Checkpoint" \
  train=False trainer.accelerator=gpu trainer.devices=1 \
  decoder=ctc_greedy


#### TensorBoard
Run this after training to inspect logs.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs
